# LAB3 — Bài 4: Iris với 10-fold cross-validation

**Sinh viên:** Trần Chí Nguyên — **MSSV:** 6451071052.

> Mỗi câu lập trình in họ tên và MSSV theo template. Chạy lần lượt từ trên xuống.

In [1]:
%matplotlib inline
HO_TEN = 'Trần Chí Nguyên'
MSSV = '6451071052'
def in_danh_tinh(): print(f'{HO_TEN} - {MSSV}')


## Câu a–b. Đọc và chuẩn hóa Iris

Đọc Iris từ sklearn và chia từng đặc trưng cho giá trị lớn nhất cột đó.

In [2]:
in_danh_tinh()
import numpy as np
from sklearn.datasets import load_iris
iris=load_iris(); X=iris.data/iris.data.max(axis=0); y=iris.target
print('X:',X.shape,'y:',y.shape,'classes:',iris.target_names)

Trần Chí Nguyên - 6451071052


X: (150, 4) y: (150,) classes: ['setosa' 'versicolor' 'virginica']


## Câu c–d. Định nghĩa mô hình và huấn luyện 10 folds

Mạng Dense(10,20,3), Adam, sparse categorical crossentropy, accuracy. In Accuracy/Precision/Recall/F1 từng fold và lưu từng model.

In [3]:
in_danh_tinh()
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from pathlib import Path
def create_model():
    m=Sequential([Dense(10,activation='relu',input_shape=(4,)),Dense(20,activation='relu'),Dense(3,activation='softmax')])
    m.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy']); return m
cv=StratifiedKFold(n_splits=10,shuffle=True,random_state=42)
models=[]; fold_metrics=[]; p_by_class=[]; r_by_class=[]; f_by_class=[]
model_dir=Path('outputs/iris_kfold'); model_dir.mkdir(parents=True,exist_ok=True)
for fold,(tr,va) in enumerate(cv.split(X,y),1):
    model=create_model(); model.fit(X[tr],y[tr],epochs=100,batch_size=16,verbose=0)
    pred=np.argmax(model.predict(X[va],verbose=0),axis=1)
    metrics={'accuracy':accuracy_score(y[va],pred),'precision':precision_score(y[va],pred,average='macro',zero_division=0),
             'recall':recall_score(y[va],pred,average='macro',zero_division=0),'f1':f1_score(y[va],pred,average='macro',zero_division=0)}
    print(f'Fold {fold}: '+', '.join(f'{k}={v:.4f}' for k,v in metrics.items()))
    fold_metrics.append(metrics); models.append(model); model.save(model_dir/f'iris_fold_{fold}.keras')
    p_by_class.append(precision_score(y[va],pred,labels=[0,1,2],average=None,zero_division=0))
    r_by_class.append(recall_score(y[va],pred,labels=[0,1,2],average=None,zero_division=0))
    f_by_class.append(f1_score(y[va],pred,labels=[0,1,2],average=None,zero_division=0))

Trần Chí Nguyên - 6451071052


C:\Users\Nguyen\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Fold 1: accuracy=1.0000, precision=1.0000, recall=1.0000, f1=1.0000


C:\Users\Nguyen\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Fold 2: accuracy=1.0000, precision=1.0000, recall=1.0000, f1=1.0000


C:\Users\Nguyen\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


... (11 outputs left out)

C:\Users\Nguyen\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Fold 9: accuracy=0.9333, precision=0.9444, recall=0.9333, f1=0.9327


C:\Users\Nguyen\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Fold 10: accuracy=1.0000, precision=1.0000, recall=1.0000, f1=1.0000


## Câu e–f. Tổng hợp trung bình và độ đo từng lớp

In trung bình macro qua folds và trung bình Precision/Recall/F1 của từng lớp.

In [4]:
in_danh_tinh()
for metric in ['accuracy','recall','precision','f1']:
    vals=[m[metric] for m in fold_metrics]; print(f'Average {metric}: {np.mean(vals):.4f} (std {np.std(vals):.4f})')
for i,name in enumerate(iris.target_names):
    print(f'{name}: Recall={np.mean(np.array(r_by_class)[:,i]):.4f}, Precision={np.mean(np.array(p_by_class)[:,i]):.4f}, F1={np.mean(np.array(f_by_class)[:,i]):.4f}')

Trần Chí Nguyên - 6451071052
Average accuracy: 0.9667 (std 0.0537)
Average recall: 0.9667 (std 0.0537)
Average precision: 0.9716 (std 0.0468)
Average f1: 0.9660 (std 0.0548)
setosa: Recall=1.0000, Precision=1.0000, F1=1.0000
versicolor: Recall=0.9200, Precision=0.9800, F1=0.9439
virginica: Recall=0.9800, Precision=0.9348, F1=0.9542


## Câu g. Dự đoán ba mẫu mới bằng mô hình F1 tốt nhất

Chọn model có macro F1 cao nhất và phân lớp ba vector yêu cầu.

In [5]:
in_danh_tinh()
best_idx=int(np.argmax([m['f1'] for m in fold_metrics])); best_model=models[best_idx]
new=np.array([[6.2,2.9,4.3,1.3],[5.1,3.5,1.4,0.2],[7.3,2.8,6.4,2.1]])/iris.data.max(axis=0)
pred=np.argmax(best_model.predict(new,verbose=0),axis=1)
print('Fold được chọn:',best_idx+1,'F1:',fold_metrics[best_idx]['f1'])
print('Nhãn dự đoán:',iris.target_names[pred])

Trần Chí Nguyên - 6451071052


Fold được chọn: 1 F1: 1.0
Nhãn dự đoán: ['versicolor' 'setosa' 'virginica']
